In [11]:
# %%
import numpy as np
import pandas as pd

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.kiselev.lesson3 import Exercise

In [ ]:
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(int)

n_classes = len(np.unique(y))
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Features: {X.shape[1]}, Classes: {n_classes}")

X shape: (1797, 64), y shape: (1797,)
Features: 64 (8x8), Classes: 10


In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y,
    shuffle=True,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp,
    shuffle=True,
)

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Val:   {len(X_val)} ({len(X_val)/len(X)*100:.0f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")

Train: 1078 (60%)
Val:   359 (20%)
Test:  360 (20%)


In [14]:
X_train_norm = X_train / 16.0
X_val_norm = X_val / 16.0
X_test_norm = X_test / 16.0

print(f"Train mean={X_train_norm.mean():.4f}, std={X_train_norm.std():.4f}")

Train mean=0.3056, std=0.3761


In [15]:

def build_model(hidden_size: int, rng: np.random.Generator):
    return Exercise.create_model(
        Exercise.create_linear_layer(64, hidden_size, rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(hidden_size, 10, rng),
        Exercise.create_logsoftmax_layer(),
    )

def evaluate_model(lr, batch_size, hidden_size, X_train, y_train, X_val, y_val, n_epochs=30, seed=42):
    rng = np.random.default_rng(seed)

    model = build_model(hidden_size, rng)
    loss = Exercise.create_nll_loss()

    Exercise.train_model(model, loss, X_train, y_train, lr=lr, n_epoch=n_epochs, batch_size=batch_size)

    val_log_probs = model.forward(X_val)
    val_pred = np.argmax(val_log_probs, axis=1)
    val_acc = float(np.mean(val_pred == y_val))
    return val_acc

In [17]:

lr_list = [0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
batch_list = [4, 8, 16, 32, 64, 128]
hidden_list = [16, 32, 64, 96, 128]
seeds = [7, 42, 123]

n_epochs = 30

results = []
best = None
best_score = -1.0

total = len(lr_list) * len(batch_list) * len(hidden_list)
count = 0

for lr in lr_list:
    for bs in batch_list:
        for hidden in hidden_list:
            count += 1

            scores = []
            for seed in seeds:
                acc = evaluate_model(lr, bs, hidden, X_train_norm, y_train, X_val_norm, y_val, n_epochs=n_epochs, seed=seed)
                scores.append(acc)

            mean_acc = float(np.mean(scores))
            std_acc = float(np.std(scores))
            stability = mean_acc - std_acc

            results.append({
                "lr": float(lr),
                "batch_size": int(bs),
                "hidden_size": int(hidden),
                "mean_acc": mean_acc,
                "std_acc": std_acc,
                "stability": stability,
                "min_acc": float(np.min(scores)),
                "max_acc": float(np.max(scores)),
            })

            if stability > best_score:
                best_score = stability
                best = {"lr": float(lr), "batch_size": int(bs), "hidden_size": int(hidden)}

df = pd.DataFrame(results).sort_values(["stability", "mean_acc"], ascending=False)

print(df.head(10).to_string(index=False))

print("\nBest params:", best, "best stability:", best_score)


TOP-10 by stability (mean - std):
  lr  batch_size  hidden_size  mean_acc  std_acc  stability  min_acc  max_acc
0.10           4           32  0.963788 0.002274   0.961514 0.961003 0.966574
0.03           8          128  0.961003 0.000000   0.961003 0.961003 0.961003
0.10           4          128  0.964717 0.004734   0.959982 0.958217 0.969359
0.10           4           64  0.961931 0.002626   0.959305 0.958217 0.963788
0.10          32           96  0.961931 0.002626   0.959305 0.958217 0.963788
0.01           4           96  0.960074 0.001313   0.958761 0.958217 0.961003
0.01           4          128  0.960074 0.001313   0.958761 0.958217 0.961003
0.05          16           96  0.960074 0.001313   0.958761 0.958217 0.961003
0.10          16          128  0.960074 0.001313   0.958761 0.958217 0.961003
0.03           8           96  0.961003 0.002274   0.958728 0.958217 0.963788

Best params: {'lr': 0.1, 'batch_size': 4, 'hidden_size': 32} best stability: 0.9615139371004799
